In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


def orquestrador_tabela_1_distribuicao_renda():
    print("\n" + "=" * 70)
    print("TABELA 1: DISTRIBUIÇÃO DE INSCRITOS POR FAIXA DE RENDA")
    print("=" * 70)

    caminho_base = Path("../data/04_load/database/inscritos_final_limpo.parquet")
    pasta_saida = Path("../reports/figures/tabelas")
    pasta_saida.mkdir(parents=True, exist_ok=True)

    df = carregar_e_filtrar_base(caminho_base)
    df_tabela = gerar_distribuicao_por_faixa_renda(df)

    print("\nTabela 1:")
    print(df_tabela)

    plotar_tabela_1_como_imagem(
        df_tabela=df_tabela,
        pasta_saida=pasta_saida
    )


def carregar_e_filtrar_base(caminho_base):
    print("[*] Carregando base...")
    df = pd.read_parquet(str(caminho_base))

    df["modalidade_fies"] = (
        df["modalidade_fies"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    # Mesmo recorte usado nas matrizes: opção prioritária e Modalidade I.
    df = df[
        (df["opcao_curso"] == 1) &
        (df["modalidade_fies"] == "MODALIDADE I")
    ].copy()

    df["renda_per_capita"] = pd.to_numeric(
        df["renda_per_capita"],
        errors="coerce"
    )

    df = df.dropna(subset=["renda_per_capita"]).copy()

    print(f"Total após filtro: {len(df):,}".replace(",", "."))

    return df


def gerar_distribuicao_por_faixa_renda(df):
    bins_renda = [-np.inf, 600, 1200, 1800, 2400, 3000, np.inf]

    labels_renda = [
        "Até 600",
        "601–1.200",
        "1.201–1.800",
        "1.801–2.400",
        "2.401–3.000",
        "Acima de 3.000"
    ]

    df["faixa_renda"] = pd.cut(
        df["renda_per_capita"],
        bins=bins_renda,
        labels=labels_renda,
        ordered=True
    )

    df_renda = (
        df.groupby("faixa_renda", observed=True)
          .size()
          .reindex(labels_renda, fill_value=0)
          .reset_index(name="inscritos")
    )

    total = df_renda["inscritos"].sum()

    df_renda["percentual"] = df_renda["inscritos"] / total * 100
    df_renda["percentual_acumulado"] = df_renda["percentual"].cumsum()

    df_tabela = pd.DataFrame({
        "Faixa de renda per capita (R$)": df_renda["faixa_renda"].astype(str),
        "Inscritos": df_renda["inscritos"],
        "%": df_renda["percentual"],
        "% acumulado": df_renda["percentual_acumulado"]
    })

    return df_tabela


def formatar_numero_inteiro(valor):
    return f"{int(valor):,}".replace(",", ".")


def formatar_percentual(valor):
    return f"{valor:.1f}%".replace(".", ",")


def plotar_tabela_1_como_imagem(df_tabela, pasta_saida):
    # Versão formatada apenas para exibição na imagem.
    df_fmt = df_tabela.copy()

    df_fmt["Inscritos"] = df_fmt["Inscritos"].apply(formatar_numero_inteiro)
    df_fmt["%"] = df_fmt["%"].apply(formatar_percentual)
    df_fmt["% acumulado"] = df_fmt["% acumulado"].apply(formatar_percentual)

    plt.rcParams["font.family"] = "DejaVu Sans"

    # Tamanho pensado para caber bem em largura de duas colunas.
    fig, ax = plt.subplots(figsize=(7.2, 2.9), dpi=300)
    ax.axis("off")

    tabela = ax.table(
        cellText=df_fmt.values,
        colLabels=df_fmt.columns,
        loc="center",
        cellLoc="center",
        colLoc="center",
        colWidths=[0.43, 0.22, 0.15, 0.20]
    )

    tabela.auto_set_font_size(False)
    tabela.set_fontsize(10.3)
    tabela.scale(1, 1.55)

    cor_cabecalho = "#1f4e79"
    cor_linha_1 = "#eaf2f8"
    cor_linha_2 = "#ffffff"
    cor_borda = "#b7c9d6"

    for (linha, coluna), celula in tabela.get_celld().items():
        celula.set_edgecolor(cor_borda)
        celula.set_linewidth(0.8)

        if linha == 0:
            celula.set_facecolor(cor_cabecalho)
            celula.get_text().set_color("white")
            celula.get_text().set_fontweight("bold")
            celula.get_text().set_ha("center")
        else:
            celula.set_facecolor(
                cor_linha_1 if linha % 2 == 1 else cor_linha_2
            )

            if coluna == 0:
                celula.get_text().set_fontweight("bold")
                celula.get_text().set_ha("left")
            else:
                celula.get_text().set_ha("center")

    plt.tight_layout()

    caminho_png = pasta_saida / "tabela_1_distribuicao_renda.png"
    caminho_pdf = pasta_saida / "tabela_1_distribuicao_renda.pdf"

    plt.savefig(caminho_png, bbox_inches="tight", dpi=700)
    plt.savefig(caminho_pdf, bbox_inches="tight")
    plt.close()

    print(f"Tabela 1 exportada para: {caminho_png}")
    print(f"Tabela 1 em PDF exportada para: {caminho_pdf}")


orquestrador_tabela_1_distribuicao_renda()


TABELA 1: DISTRIBUIÇÃO DE INSCRITOS POR FAIXA DE RENDA
[*] Carregando base...
Total após filtro: 1.102.122

Tabela 1:
  Faixa de renda per capita (R$)  Inscritos          %  % acumulado
0                        Até 600     320976  29.123455    29.123455
1                      601–1.200     450100  40.839399    69.962853
2                    1.201–1.800     199204  18.074587    88.037441
3                    1.801–2.400      74404   6.750977    94.788417
4                    2.401–3.000      48939   4.440434    99.228851
5                 Acima de 3.000       8499   0.771149   100.000000
Tabela 1 exportada para: ../reports/figures/tabelas/tabela_1_distribuicao_renda.png
Tabela 1 em PDF exportada para: ../reports/figures/tabelas/tabela_1_distribuicao_renda.pdf
